# Simplicial Complexes and the Vietoris-Rips Construction

This notebook introduces:
1. **Simplicial complexes** -- the combinatorial building blocks of TDA
2. **Vietoris-Rips (VR) complex** -- the most common filtration for point cloud data
3. Hands-on construction using `gudhi` and visualisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

try:
    import gudhi
    HAS_GUDHI = True
except ImportError:
    HAS_GUDHI = False
    print('gudhi not installed -- pip install gudhi')

%matplotlib inline
plt.rcParams['figure.figsize'] = (7, 7)

## 1. What is a Simplicial Complex?

A **simplex** of dimension $k$ is the convex hull of $k+1$ affinely independent points:
- 0-simplex = vertex, 1-simplex = edge, 2-simplex = triangle, 3-simplex = tetrahedron.

A **simplicial complex** $K$ is a collection of simplices closed under taking faces:
if $\sigma \in K$ and $\tau \subseteq \sigma$, then $\tau \in K$.

In [ ]:
# Manual construction of a small simplicial complex
# Vertices: 0, 1, 2, 3
# Edges: {0,1}, {1,2}, {0,2}, {2,3}
# Triangle: {0,1,2}

vertices = [0, 1, 2, 3]
edges = [(0,1), (1,2), (0,2), (2,3)]
triangles = [(0,1,2)]

# Verify closure property: all faces of the triangle are edges
for tri in triangles:
    for face in combinations(tri, 2):
        assert face in edges or tuple(reversed(face)) in edges, f"Missing face {face}"
print("Simplicial complex is valid (closed under faces).")

# Euler characteristic: V - E + F
chi = len(vertices) - len(edges) + len(triangles)
print(f"Euler characteristic: {chi}")

## 2. Vietoris-Rips Complex from a Point Cloud

Given a point cloud $P \subset \mathbb{R}^d$ and a scale parameter $\varepsilon > 0$:
$$\text{VR}(P, \varepsilon) = \{\sigma \subseteq P : \text{diam}(\sigma) \le \varepsilon\}$$

As $\varepsilon$ grows from 0, the complex gains simplices -- this is the **Rips filtration**.

In [ ]:
# Sample points on a noisy circle
np.random.seed(42)
n = 30
theta = np.linspace(0, 2*np.pi, n, endpoint=False)
noise = 0.1 * np.random.randn(n, 2)
points = np.column_stack([np.cos(theta), np.sin(theta)]) + noise

plt.scatter(points[:, 0], points[:, 1], s=40, zorder=5)
plt.axis('equal')
plt.title('Point Cloud (noisy circle)')
plt.show()

In [ ]:
# Visualise the VR complex at different epsilon values
from scipy.spatial.distance import pdist, squareform

D = squareform(pdist(points))

def draw_vr(points, D, eps, ax):
    n = len(points)
    # Draw edges
    for i in range(n):
        for j in range(i+1, n):
            if D[i, j] <= eps:
                ax.plot(*zip(points[i], points[j]), 'b-', alpha=0.3, linewidth=0.8)
    # Draw triangles
    for i in range(n):
        for j in range(i+1, n):
            for k in range(j+1, n):
                if D[i,j] <= eps and D[i,k] <= eps and D[j,k] <= eps:
                    tri = plt.Polygon(points[[i,j,k]], alpha=0.08, color='blue')
                    ax.add_patch(tri)
    ax.scatter(points[:, 0], points[:, 1], s=20, c='red', zorder=5)
    ax.set_title(f'VR complex, eps={eps:.2f}')
    ax.set_aspect('equal')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, eps in zip(axes, [0.3, 0.6, 1.0]):
    draw_vr(points, D, eps, ax)
plt.tight_layout()
plt.show()

In [ ]:
# Build VR complex with GUDHI
if HAS_GUDHI:
    rips = gudhi.RipsComplex(points=points, max_edge_length=1.5)
    simplex_tree = rips.create_simplex_tree(max_dimension=2)
    
    print(f"Number of vertices:  {simplex_tree.num_vertices()}")
    print(f"Number of simplices: {simplex_tree.num_simplices()}")
    
    # Show first 10 simplices with their filtration values
    for i, (simplex, filt) in enumerate(simplex_tree.get_filtration()):
        if i >= 10:
            break
        print(f"  {simplex}  filtration = {filt:.3f}")
else:
    print('Install gudhi to build the VR complex programmatically.')

## 3. Betti Numbers

The **Betti numbers** $\beta_k$ count the number of $k$-dimensional holes:
- $\beta_0$ = connected components
- $\beta_1$ = loops / tunnels
- $\beta_2$ = voids

For our circle, we expect $\beta_0 = 1$ (one component) and $\beta_1 = 1$ (one loop) at the right scale.

In [ ]:
if HAS_GUDHI:
    simplex_tree.compute_persistence()
    betti = simplex_tree.betti_numbers()
    print(f"Betti numbers: {betti}")
    print(f"  beta_0 = {betti[0]} (connected components)")
    if len(betti) > 1:
        print(f"  beta_1 = {betti[1]} (loops)")

## Key Takeaways

- A **simplicial complex** generalises a graph to higher dimensions.
- The **Vietoris-Rips filtration** builds a nested family of complexes parametrised by scale $\varepsilon$.
- Topological features (components, loops, voids) emerge and disappear as $\varepsilon$ varies.

**Next:** Persistent homology -- tracking these features across scales.